<a href="https://colab.research.google.com/github/Manoj-Kumar-Ejjada-AI/models/blob/manoj_models/GenAI/Cross_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [22]:
class CrossAttention(nn.Module):
  def __init__(self,
               dim_model,
               row_dim,
               col_dim):
    super().__init__()

    self.row_dim = row_dim
    self.col_dim = col_dim

    self.w_q = nn.Linear(in_features=dim_model, out_features=dim_model, bias = False)
    self.w_k = nn.Linear(in_features=dim_model, out_features=dim_model, bias = False)
    self.w_v = nn.Linear(in_features=dim_model, out_features=dim_model, bias = False)

  def forward(self,
              encodings_for_q,
              encodings_for_k,
              encodings_for_v, mask = None):

    q = self.w_q(encodings_for_q)
    k = self.w_k(encodings_for_k)
    v = self.w_v(encodings_for_v)

    similarities = torch.matmul(q,k.transpose(dim0 = self.row_dim, dim1 = self.col_dim))
    scaled_sims = similarities/(torch.tensor(k.size(self.col_dim)**0.5))

    if mask is not None:
      scaled_sims = scaled_sims.masked_fill(mask, value = -1e9)

    percents = F.softmax(scaled_sims,dim = self.col_dim)
    attention_scores = torch.matmul(percents,v)
    return attention_scores



In [23]:
## create matrices of token encodings
encodings_for_q = torch.tensor([[1.16, 0.23],
                                [0.57, 1.36],
                                [4.41, -2.16]])

encodings_for_k = torch.tensor([[1.16, 0.23],
                                [0.57, 1.36],
                                [4.41, -2.16]])

encodings_for_v = torch.tensor([[1.16, 0.23],
                                [0.57, 1.36],
                                [4.41, -2.16]])

## set the seed for the random number generator
torch.manual_seed(42)

## create an attention object
attention = CrossAttention(dim_model=2,
                      row_dim=0,
                      col_dim=1)

## calculate encoder-decoder attention
attention(encodings_for_q, encodings_for_k, encodings_for_v)

tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)